In [3]:
# ── Cell 1 — Configuration ─────────────────────────────────────────────────────

DATASET    = "princeton-nlp/SWE-bench_Verified"
OUTPUT_FILE = "pilot_instances.json"

N_SINGLE    = 25   # single-file instances per batch
N_MULTI     = 25   # multi-file instances per batch
MAX_BATCHES = 2   # total batches to pre-sample (increase if needed)

RANDOM_SEED = 42   # fix this forever — changing it invalidates all past results


In [4]:
# ── Cell 2 — Load dataset ──────────────────────────────────────────────────────
from datasets import load_dataset

ds = load_dataset(DATASET, split="test")
print(f"Loaded {len(ds)} instances")


Loaded 500 instances


In [5]:
# ── Cell 3 — Split by file count ───────────────────────────────────────────────

def count_files(inst):
    """Number of distinct files touched by the gold patch."""
    return sum(1 for line in inst["patch"].splitlines() if line.startswith("--- a/"))

all_instances = list(ds)
single_pool   = [i for i in all_instances if count_files(i) == 1]
multi_pool    = [i for i in all_instances if count_files(i) >  1]

print(f"Single-file pool : {len(single_pool)}")
print(f"Multi-file pool  : {len(multi_pool)}")
print(f"Can support up to {min(len(single_pool)//N_SINGLE, len(multi_pool)//N_MULTI)} batches ")


Single-file pool : 429
Multi-file pool  : 71
Can support up to 2 batches 


In [6]:
# ── Cell 4 — Sample & save ─────────────────────────────────────────────────────
import random, json

random.seed(RANDOM_SEED)

all_single = random.sample(single_pool, N_SINGLE * MAX_BATCHES)
all_multi  = random.sample(multi_pool,  N_MULTI  * MAX_BATCHES)

batches = []
for b in range(MAX_BATCHES):
    batch = (
        all_single[b * N_SINGLE : (b + 1) * N_SINGLE]
        + all_multi [b * N_MULTI  : (b + 1) * N_MULTI ]
    )
    random.shuffle(batch)
    batches.append([i["instance_id"] for i in batch])

with open(OUTPUT_FILE, "w") as f:
    json.dump(batches, f, indent=2)

print(f"Saved {MAX_BATCHES} batches of {N_SINGLE + N_MULTI} → {OUTPUT_FILE}")
for b, batch in enumerate(batches):
    print(f"  Batch {b}: {batch[:3]} ... ({len(batch)} instances)")


Saved 2 batches of 50 → pilot_instances.json
  Batch 0: ['astropy__astropy-14995', 'django__django-13810', 'sphinx-doc__sphinx-8595'] ... (50 instances)
  Batch 1: ['sympy__sympy-13877', 'django__django-12741', 'matplotlib__matplotlib-23412'] ... (50 instances)


In [7]:
# ── Cell 5 — Clone repos ─────────────────────────────────────────────────────────
import subprocess, os

REPOS_BASE = "repos"

# (org, repo) pairs for all SWE-bench Verified repositories
REPOS = [
    ("astropy",      "astropy"),
    ("django",       "django"),
    ("matplotlib",   "matplotlib"),
    ("pylint-dev",   "pylint"),
    ("pytest-dev",   "pytest"),
    ("scikit-learn", "scikit-learn"),
    ("sphinx-doc",   "sphinx"),
    ("sympy",        "sympy"),
    ("pydata",       "xarray"),
]

os.makedirs(REPOS_BASE, exist_ok=True)

for org, repo in REPOS:
    dest = os.path.join(REPOS_BASE, repo)
    if os.path.isdir(dest):
        print(f"  already exists — skipping {repo}")
        continue
    url = f"https://github.com/{org}/{repo}.git"
    print(f"  cloning {url} → {dest} ...")
    r = subprocess.run(
        ["git", "clone", "--filter=blob:none", url, dest],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f"  ✓ {repo}")
    else:
        print(f"  ✗ {repo}: {r.stderr.strip()}")

print("\nDone.")

  already exists — skipping astropy
  already exists — skipping django
  already exists — skipping matplotlib
  already exists — skipping pylint
  already exists — skipping pytest
  already exists — skipping scikit-learn
  already exists — skipping sphinx
  already exists — skipping sympy
  already exists — skipping xarray

Done.
